In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import pandas as pd
from tqdm import tqdm
from utils import combined_approaches as ca
from utils import global_strategies as gs
from utils import label_based_measures as lbm
from utils import local_single_attribute as lsa
from utils import similarity_structures as ss
from utils import top_down_data_structures as tdds
from utils import value_overlap as vo
from utils import evaluation as eval

# ESERCIZIO DATA INTEGRATION - TOP DOWN

Considerare:

In [3]:
path=''
path="http://dbgroup.ing.unimore.it/EBI/TopC/"

src_links = [
  path + 'S1.csv',
  path + 'S2.csv',
  path + 'S3.csv'   ]

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv(path +'GoldStandard.csv').astype(str)
GlobalSchema=pd.read_csv(path +'GlobalSchema.csv').astype(str)
tdds.to_GMM(GoldStandard)

SOURCE,S1,S2,S3
GAT,,,
author1,[Writer],[PrimoAutoreDelLibro],[Creator]
author2,[Writer],[SecondoAutoreDelLibro],[]
author3,[],[TerzoAutoreDelLibro],[]
nookbookprice,[price],[],[]
paperbackprice,[price],[],[]
publicationdate,[PublicationInfo],[DataDiPubblicazione],[]
publisher,[PublicationInfo],[],[distributor]
ratingscount,[],[Valutazione],[number of votes]
ratingvalue,[],[Valutazione],[number of votes]


In [7]:
ValutazioneMatchTable = pd.DataFrame(columns=['MT','TP','FP','FN','P','R','F'])

In [32]:
# Data
def CalcoloMatchingTable(TableL:pd.DataFrame,TableR:pd.DataFrame):

        SimTableA = lbm.levenshtein_label_based_similarity(TableL, TableR)
        #SimTableB = lbm.jaro_label_based_similarity(TableL, TableR)
        #SimTableB = vo.value_overlap_sim(GlobalSchema, Sources[y])
        SimTableC= vo.value_overlap_simjoin_jaccard(TableL, TableR, 0.3)

        # combiner
        SimTable = ca.avg_sim_table([SimTableA,SimTableC])
        #SimTable = ca.min_sim_table([SimTableA,SimTableC])
        #SimTable = ca.max_sim_table([SimTableC,SimTableA])

        # Weighted-sum
        # SimTable = ca.Weighted_sum([SimTableA,SimTableB,SimTableC], [.3,.4,.3] )


        # dalla tabella di similarità alle corrispondenze

        MatchTable= lsa.thresholding(SimTable, 0.5)

        MatchTable = lsa.top_K(MatchTable,1,'A')
        #MatchTable = lsa.top_K(MatchTable,3, 'B')

         # global mapping
        #MatchTable = gs.stable_marriage(MatchTable)
        #MatchTable = gs.simmetric_best_match(MatchTable)

        return MatchTable

In [33]:
def CalcoloGlobalMatchingTable(Sources, GlobalSchema:pd.DataFrame):
    GlobalMatchingTable = pd.DataFrame(columns=['GAT','SOURCE','LAT','SLAT','sim'])
    for y in tqdm(Sources.keys()):
        MatchTable = CalcoloMatchingTable(GlobalSchema, Sources[y])
        
        MatchTable.columns = ['GAT','LAT','sim']
        MatchTable['SOURCE'] = str(y)
        MatchTable['SLAT'] = MatchTable['SOURCE']+'_'+MatchTable['LAT']
        GlobalMatchingTable = GlobalMatchingTable.append(MatchTable, sort=False)

    return GlobalMatchingTable

## Discussione

Lo svolgimento/risposta consiste nella discussione strutturata nei seguenti punti

1. Analizzare il gold standard dato (GoldStandard), stabilire il tipo di matching tra  il GlobalSchema e i  local schemata


2. Considerare la funzione CalcoloMatchingTable data, fare sua valutazione  e analizzare gli eventuali FP e/o FN


3. Modificare la funzione CalcoloMatchingTable,  mantendendo traccia delle modifiche e commentando brevemente i cambiamenti effettuati discutendo i   risultati ottenuti




In [12]:
eval.AnalisiGlobalMatchTable(GoldStandard, SOURCES)

1) Le seguenti SOURCE di GMT non sono definite in Sources: []
2) I seguenti SLAT di GMT non sono definiti in Sources: []
3) GAT mappati da una sola SOURCE: ['author3', 'nookbookprice', 'paperbackprice']
4) GAT mappati in più LAT (per ciascuna SOURCE):
5) LAT mappati in più GAT (per ciascuna SOURCE):
   SOURCE: S1, LAT: PublicationInfo, GAT diversi: 2
   SOURCE: S1, LAT: Writer, GAT diversi: 2
   SOURCE: S1, LAT: price, GAT diversi: 2
   SOURCE: S2, LAT: Valutazione, GAT diversi: 2
   SOURCE: S3, LAT: number of votes, GAT diversi: 2


In [34]:
GMTcalcolata=CalcoloGlobalMatchingTable(SOURCES, GlobalSchema)
tdds.to_GMM(GMTcalcolata)

100%|██████████| 3/3 [00:02<00:00,  1.04it/s]


SOURCE,S1,S2,S3
GAT,,,
author1,[Writer],[PrimoAutoreDelLibro],[Creator]
author2,[Writer],[SecondoAutoreDelLibro],[]
author3,[],[TerzoAutoreDelLibro],[]
nookbookprice,[price],[],[]
paperbackprice,[price],[],[]
publicationdate,[PublicationInfo],[DataDiPubblicazione],[]
publisher,[PublicationInfo],[],[distributor]
ratingscount,[],[Valutazione],[number of votes]
ratingvalue,[],[Valutazione],[number of votes]


In [8]:
print(eval.Valuta(GoldStandard[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']]))
print(eval.Vedi_Valuta(GoldStandard[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']], 'FN'))
print(eval.Vedi_Valuta(GoldStandard[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']], 'FP'))

   MT  TP  FP  FN       P       R       F
0  17  14   3   5  0.8235  0.7368  0.7778
                 A                   B     _merge
1          author2           S1_Writer  left_only
5        publisher  S1_PublicationInfo  left_only
6   paperbackprice            S1_price  left_only
12     ratingvalue      S2_Valutazione  left_only
17     ratingvalue  S3_number of votes  left_only
              A               B      _merge
19        pages        S2_pages  right_only
20        pages        S3_pages  right_only
21  ratingvalue  S3_ratingvalue  right_only


In [35]:
X = eval.Valuta(GoldStandard[['GAT', 'SLAT']],GMTcalcolata[['GAT', 'SLAT']])
X

,MT,TP,FP,FN,P,R,F
0,19,19,0,0,1.0,1.0,1.0


In [36]:
ValutazioneMatchTable = ValutazioneMatchTable.append(X).rename(index={0: "Label-Instance-Avg-Top1-NoGlobalMapping"})
ValutazioneMatchTable

,MT,TP,FP,FN,P,R,F
Label-Instance-Max-0.5-StableMarriage,17,14,3,5,0.8235,0.7368,0.7778
Label-Instance-Avg-0.5-StableMarriage,14,14,0,5,1.0000,0.7368,0.8485
Label-Instance-Avg-0.4-StableMarriage,17,14,3,5,0.8235,0.7368,0.7778
Label-Instance-Avg-0.4-StableMarriage,13,13,0,6,1.0000,0.6842,0.8125
Label-Instance-Avg-Top1-NoGlobalMapping,19,19,0,0,1.0000,1.0000,1.0000
